# Dual-axis Gini: fixing the hub-feature problem without losing steering impact

Recap: per-sample (row) Gini alone produces polysemantic hub features (top-10%-by-importance purity ~0.001). The MoE-style load-balance fix (`gini_load_balanced.py`) restored purity but pushed feature usage toward uniform, which caps per-occurrence steering impact -- it fell to ~45% of TopK's, well below the original (broken) Gini's 2x-over-TopK number.

`gini_dual_axis.py` instead maximizes Gini along BOTH axes of the latent code matrix: rows (few features per sample, as before) and columns (each feature fires on few samples, hard). A hub feature -- uniformly large across every sample -- is the global *minimum* of column Gini, so maximizing column Gini structurally forbids hubs while explicitly rewarding rare-but-strong features, which is exactly the property that drives steering impact. This sweeps `lambda_col` and reports purity, importance concentration, and steering impact together, alongside a sparsity-matched TopK reference trained in the same run.

**Win condition:** a `lambda_col` where purity stays well above the broken baseline (ideally near or above the load-balanced fix's ~0.17-0.24 top-10%-purity range) *and* ablate/clamp climbs back above the TopK reference row (~0.09 / ~0.18).

**Failure modes to watch:** `lambda_col` too high starving reconstruction (MSE spikes, `n_valid_features` craters -- dead-feature collapse), or the row/column terms fighting so sparsity drifts away from the TopK reference's, breaking the matched comparison.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# lambda_col=0.0 is the control (equivalent to the original row-only Gini,
# lambda_row=0.1). The run also trains a TopK reference matched to the
# lambda_col=0.0 sparsity, so the comparison target is consistent across
# every lambda_col value.
!python gini_dual_axis.py --dataset fashion_mnist --seed 0 --lambda-row 0.1 --lambda-cols 0.0 0.01 0.05 0.1 0.3

In [ ]:
import json
with open('results/gini_dual_axis/fashion_mnist_seed0.json') as f:
    results = json.load(f)

print(f"{'Model':22s} {'MSE':>8s} {'Sparsity':>9s} {'ImpGini':>8s} {'Top10Share':>11s} "
      f"{'MeanPurity':>11s} {'Top10Purity':>12s} {'Ablate':>8s} {'Clamp':>8s}")
for name, r in results.items():
    print(f"{name:22s} {r['mse']:8.4f} {r['relative_sparsity']:9.3f} "
          f"{r['importance_gini_coefficient']:8.3f} {r['top10_importance_share']:11.3f} "
          f"{r['mean_purity']:11.3f} {r['mean_purity_top10pct_by_importance']:12.3f} "
          f"{r['steering_impact_ablate']:8.4f} {r['steering_impact_clamp']:8.4f}")
results

In [ ]:
!zip -r gini_dual_axis_results.zip results/gini_dual_axis
from google.colab import files
files.download('gini_dual_axis_results.zip')